In [2]:
from tqdm import tqdm
import requests
import zipfile
import os
import time
import hashlib

# Example list of presigned URLs for multipart ZIP files
presigned_urls = ['https://brage.it.ntnu.no/mhbucket/CMIC/raw_data/saramorg_fret_nroot.zip?AWSAccessKeyId=AKIA8E210FD3550BED0D&Signature=uiNok50lAendE6jRc76Y1KVBL3M%3D&Expires=1778851312']

def download_and_extract_file(url, filename, extract_to, retries=5, backoff_factor=1.0):
    """Download an individual file to disk with retry logic, calculate its MD5, and extract it to the target directory."""
    attempt = 0
    while attempt < retries:
        try:
            response = requests.get(url, stream=True, timeout=(10, 60))  # Connect and read timeout
            response.raise_for_status()  # Check for HTTP errors

            # Setup progress bar
            total_size = int(response.headers.get('content-length', 0))
            block_size = 1024  # 1 Kibibyte
            progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc=f"Downloading {filename}")
            
            # Initialize MD5 hash calculator
            md5_hash = hashlib.md5()

            # Write file to disk and update MD5 hash
            with open(filename, 'wb') as file:
                for data in response.iter_content(block_size):
                    progress_bar.update(len(data))
                    file.write(data)
                    md5_hash.update(data)  # Update MD5 hash with chunk of data
            progress_bar.close()

            # Display the calculated MD5 hash
            file_md5 = md5_hash.hexdigest()
            print(f"MD5 hash of {filename}: {file_md5}")

            # Extract the ZIP file to the specified directory (repo root)
            print(f"Extracting {filename} to {extract_to} ...")
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            print(f"{filename} extracted to {extract_to}.")

            # Remove the ZIP file after extraction
            os.remove(filename)
            return  # Exit function after successful download, hash calculation, and extraction
        
        except (requests.exceptions.RequestException, requests.exceptions.Timeout) as e:
            print(f"Download error: {e}, retrying in {backoff_factor * (2 ** attempt)} seconds...")
            time.sleep(backoff_factor * (2 ** attempt))
            attempt += 1

    print(f"Failed to download and extract the file after {retries} attempts.")

# Determine the directory where this notebook resides
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.path.dirname(os.getcwd())

# Calculate target extraction directory as *this* saramorg_fret_nroot repo directory (not parent)
repo_root = notebook_dir

# Create directories for ZIP parts and extracted files
temp_dir = os.path.join(notebook_dir, "temp_zip_parts")
os.makedirs(temp_dir, exist_ok=True)

# Download, extract, and clean up each ZIP file
for index, url in enumerate(presigned_urls, start=1):
    part_filename = os.path.join(temp_dir, f"part_{index}.zip")
    download_and_extract_file(url, part_filename, extract_to=repo_root)

# Final cleanup: Remove the temporary directory (should be empty by now)
os.rmdir(temp_dir)

print(f"All files downloaded, extracted to {repo_root}, and cleaned up.")


MD5 hash of c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot\temp_zip_parts\part_1.zip: 699255fc513ede64ba620c581ef7601e
Extracting c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot\temp_zip_parts\part_1.zip to c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot ...
c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot\temp_zip_parts\part_1.zip extracted to c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot.
All files downloaded, extracted to c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot, and cleaned up.
